# Random Forest Example: Predicting Wide Receiver Draft Round

If you haven't seen yet, check out the decision tree example before continuing on in this notebook.

In this notebook we will attempt to predict wide receiver draft round based on the same stats as the decision tree example. This will allow us to compare the effectiveness of a single decision tree model and a random forest model.

We hypothesize that the random forest will perform better, but is also likely limited due to the limited scope of our analysis. By using just statistical performance measures, we can better understand how basic box score stats impact draft status. However, this fails to account for so many context and player skill factors that likely impact the draft in a much greater fashion.

In [9]:
# Import necessary libraries
import sys
import os

# Send Python to the project root so we can import our library
project_root = "/Users/maxbasurto/Documents/Rice Classes/CMOR 438/CMOR438-Project/src"
sys.path.append(project_root)

# Import our ML library and other necessary libraries
import rice_ml
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [10]:
# Import the dataset
data_path = "/Users/maxbasurto/Documents/Rice Classes/CMOR 438/CMOR438-Project/data/nfl_draft_data.csv"
data = pd.read_csv(data_path)

Now that we have the data and package imported, we can continue as we did in the decision tree notebook.

In [11]:
# Filter the dataset to include only the relevant columns
data = data[["season", "pfr_player_name", "round", "position", "receptions","rec_yards","rec_tds"]]

# Only want Wide Receivers
data = data[data["position"] == "WR"]

# Only want players drafted this century
data = data[data["season"] >= 2000]
# Now we don't need the season column
data = data.drop(columns=["season"])

# Add column for yards per reception
data["yards_per_reception"] = data["rec_yards"] / data["receptions"]
# Add column for touchdowns per reception
data["tds_per_reception"] = data["rec_tds"] / data["receptions"]

# Drop rows with missing values
data = data.dropna()

As explained in the decision tree notebook, we clean the data to better reflect the current state of the NFL and select only the necessary features and players.

In [12]:
# Standardize the data using our StandardScaler class from our library
scaler = rice_ml.StandardScaler()
scaled_data = scaler.fit_transform(data[["yards_per_reception", "tds_per_reception", "receptions", "rec_yards", "rec_tds"]])

# Combine scaled features with the target variable (round) into a new DataFrame
scaled_data = pd.DataFrame(scaled_data, columns=["yards_per_reception", "tds_per_reception", "receptions", "rec_yards", "rec_tds"])
scaled_data["round"] = data["round"].values

# Create a 80/20 train/test split with random shuffling
scaled_data = scaled_data.sample(frac=1, random_state=42).reset_index(drop=True)
train_size = int(0.8 * len(scaled_data))
train_data = scaled_data.iloc[:train_size]
test_data = scaled_data.iloc[train_size:]

# Convert pandas DataFrames to numpy arrays for training and testing
X_train = train_data[["yards_per_reception", "tds_per_reception", "receptions", "rec_yards", "rec_tds"]].values
y_train = train_data["round"].values
X_test = test_data[["yards_per_reception", "tds_per_reception", "receptions", "rec_yards", "rec_tds"]].values
y_test = test_data["round"].values

# Train a random forest classifier using our library
model = rice_ml.RandomForest(n_estimators=100, max_depth=5, min_samples_split=10)
model.train(X_train, y_train)

Now we have successfully trained our model. Note that this took significantly longer (12s) than the single decision tree, which makes sense due to the random forests' increased complexity. Now we shift to making predictions and evaluating model effectiveness.

In [15]:
# Make predictions on the test set
predictions = model.predict(X_test)

# Evaluate the model's performance using accuracy and mean absolute error from our library
accuracy = rice_ml.accuracy_score(y_test, predictions)
mae = rice_ml.mean_absolute_error(y_test, predictions)
print(f"Accuracy: {accuracy:.4f}")
print(f"Mean Absolute Error: {mae:.4f}")


Accuracy: 0.2465
Mean Absolute Error: 1.7254


We see an 8% increase in accuracy over the decision tree model and .2 lower MAE. This aligns perfectly with our hypothesis because the increased model complexity resulted in better performance. However, neither model has powerful prediction power. We still have a fairly high average error, highlighting raw statistics' inability to forecast where players will be taken in the draft.

Depsite a lack of strong predicion power, it's notable that we can correctly classify almost 30% of wide receivers using the most simple box score stats in existence. This tells us that while much more of the draft is about the context around the player and their skillset, these statistics are able to capture a fair amount of a wide receiver's skillset.

Ultimately, the NFL Draft is based heavily on the eye-test, but with analytics growing, things may become more precitable using models like these. Because as teams adapt analytics, outsiders can better anticipate the teams' decisions. 